In [1]:
# -*- coding: utf-8 -*-
# 绘制所有神经元在不同行为事件下的平均trace和heatmap

import os
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

In [2]:


# ----------------------------- 基础绘图函数 -----------------------------
def plot_colomap(df, ifCenter='True', cmap='seismic', figsize=(10,8),
                 vmin=-0.5, vmax=4, ifSave=False, savePath=None, filename=None):
    plt.figure(figsize=figsize)
    if ifCenter:
        import matplotlib.colors as mcolors
        divnorm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        plt.pcolormesh(df.T, cmap=cmap, norm=divnorm)
    else:
        plt.pcolormesh(df.T, cmap=cmap)
    plt.colorbar()
    if ifSave:
        if not os.path.exists(savePath):
            os.makedirs(savePath)
        plt.savefig(os.path.join(savePath, filename))
        plt.close()
    else:
        plt.show()

def getSummary(df_toGenerateSummary):
    Summary = pd.DataFrame()
    Summary['ALL_MICE_MEAN'] = df_toGenerateSummary.mean(axis=1)
    Summary['MEAD_SEM'] = df_toGenerateSummary.mean(axis=1) - df_toGenerateSummary.sem(axis=1)
    Summary['MEADSEM'] = df_toGenerateSummary.mean(axis=1) + df_toGenerateSummary.sem(axis=1)
    Summary['SEM'] = df_toGenerateSummary.sem(axis=1)
    Summary['MAX'] = df_toGenerateSummary.max(axis=1)
    Summary['MIN'] = df_toGenerateSummary.min(axis=1)
    return Summary

def get_minMaxForPlot_One(WT_summary):
    WT = WT_summary['ALL_MICE_MEAN'].values.tolist()
    WT_SEM = WT_summary['MEAD_SEM'].values.tolist()
    WTSEM = WT_summary['MEADSEM'].values.tolist()
    vmin = min(WT_SEM) - 0.2
    vmin = math.floor(vmin*100)/100
    vmax = max(WTSEM) * 1.2
    vmax = math.ceil(vmax*100)/100
    x = WT_summary.index.values.tolist()
    return x, WT, WT_SEM, WTSEM, vmin, vmax

def plotAgeTrace_One(x, yWT, yWT_SEM, yWTSEM, ymin, ymax, cWT, cFWT,
                     xVline=None, ifSave=False, savePath=None, filename=None):
    plt.figure(figsize=(8,4))
    plt.plot(x, yWT, color=cWT, lw=2)
    if xVline is not None:
        plt.plot([xVline, xVline], [ymin*0.8, ymax*0.8], '--r', lw=2)
    else:
        plt.plot([x[int(len(x)/2)], x[int(len(x)/2)]], [ymin*0.8, ymax*0.8], '--r', lw=2)
    plt.fill_between(x, yWT_SEM, yWTSEM, color=cFWT, alpha=0.5)
    plt.ylim(ymin, ymax)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().axes.get_xaxis().set_visible(False)
    if ifSave:
        if not os.path.exists(savePath):
            os.makedirs(savePath)
        plt.savefig(os.path.join(savePath, filename))
        plt.close()
    else:
        plt.show()

# ----------------------------- 数据处理辅助函数 -----------------------------
def mixMaxScale_dff(df, feature_range=(0, 1)):
    """改进的MinMax归一化，包含数据验证"""
    if df is None or df.empty:
        print("警告: 输入数据为空，返回原始数据")
        return df
    print(f"数据形状: {df.shape}")
    try:
        df_numeric = df.apply(pd.to_numeric, errors='coerce')
        nan_columns = df_numeric.columns[df_numeric.isna().all()].tolist()
        if nan_columns:
            print(f"警告: 以下列全为NaN值: {nan_columns}")
            df_numeric = df_numeric.drop(columns=nan_columns)
        if df_numeric.empty:
            print("错误: 转换后数据为空")
            return df
        scaler = MinMaxScaler(feature_range=feature_range)
        scaler.fit(df_numeric.copy())
        dfN = pd.DataFrame(scaler.transform(df_numeric))
        dfN.columns = df_numeric.columns
        return dfN
    except Exception as e:
        print(f"缩放过程中发生错误: {e}")
        if hasattr(df, 'head'):
            print(f"数据前几行:\n{df.head()}")
        return df

def get_BonsetA_debug(df, which_status_to_check, fps, secForward=10, secBackward=10,
                      beforeB=False, baseSecond=2, intervalLength=20,
                      if_CombineAdjacentChunks=True, gapMax=10, ifNorm=True):
    from Functions_ITRT_PCA_250811_checkframeinterval_beforeafter_eventonset import check_frame_interval, find_compositeIntervals, normBA
    df_status = check_frame_interval(df, which_status_to_check, ifKeep=True)
    combinedchunks = find_compositeIntervals(df, which_status_to_check,
                                              intervalLength=intervalLength,
                                              if_CombineAdjacentChunks=if_CombineAdjacentChunks,
                                              gapMax=gapMax)
    print(f"  {which_status_to_check}: found {len(combinedchunks)} intervals")
    df_Bonset_list = []
    df_onsetA_list = []
    if len(combinedchunks) > 0:
        if len(combinedchunks) > 0:
            print(f"  First interval: {combinedchunks[0]}, df length: {len(df)}")
        for interval in combinedchunks:
            if interval[0] >= secForward*fps:
                Bonset_start = interval[0] - secForward*fps
                Bonset_end = interval[0]
                onsetA_start = interval[0]
                if interval[0] + secBackward*fps < len(df):
                    onsetA_end = interval[0] + secBackward*fps
                    if Bonset_start >= 0 and onsetA_end <= len(df):
                        Bonset_df = df_status.iloc[Bonset_start:Bonset_end]
                        Bonset_df = Bonset_df.drop('status', axis=1)
                        df_Bonset_list.append(Bonset_df)
                        onsetA_df = df_status.iloc[onsetA_start:onsetA_end]
                        onsetA_df = onsetA_df.drop('status', axis=1)
                        df_onsetA_list.append(onsetA_df)
                    else:
                        print(f"    Interval {interval} out of bounds")
                else:
                    print(f"    Interval {interval}: not enough frames after onset")
                    print(f"    Required: {interval[0] + secBackward*fps}, Available: {len(df)}")
    print(f"  Valid intervals found: {len(df_Bonset_list)} before, {len(df_onsetA_list)} after")
    if ifNorm:
        df_Bonset_listN, df_onsetA_listN = normBA(df_status, df_Bonset_list,
                                                  df_onsetA_list, beforeB=beforeB,
                                                  baseSecond=baseSecond)
    else:
        df_Bonset_listN, df_onsetA_listN = df_Bonset_list, df_onsetA_list
    return (df_Bonset_listN, df_onsetA_listN)

def average_nActEntries_debug(df_list, neuron_list, fps, seconds):
    frames = seconds*fps
    avgNeuron_actEntries = pd.DataFrame()
    print(f"  average_nActEntries: {len(df_list)} dataframes, expecting {frames} frames")
    if len(df_list) > 0:
        for nCol in neuron_list:
            nAct_entries = []
            for i in range(len(df_list)):
                if df_list[i].shape[0] == frames:
                    nAct_entries.append(df_list[i][nCol].astype(float))
                else:
                    print(f"    DataFrame {i} has shape {df_list[i].shape}, expected {frames}")
            if len(nAct_entries) > 0:
                nAct_mean = np.mean(np.array(nAct_entries), axis=0)
                avgNeuron_actEntries[nCol] = nAct_mean
            else:
                print(f"    No valid entries for neuron {nCol}")
    print(f"  Result shape: {avgNeuron_actEntries.shape}")
    return avgNeuron_actEntries

def separate_session(df, threshold=3):
    """根据时间戳分割不同session"""
    df = pd.read_csv(df, index_col=0)
    df.columns = [c.strip() for c in df.columns]
    df['index_col'] = pd.to_numeric(df.index, errors='coerce')
    check_points = [0]
    frame_list = df['index_col'].values.tolist()
    for i in range(len(frame_list) - 1):
        if (frame_list[i+1] - frame_list[i]) > threshold:
            check_points.append(frame_list[i+1])
    check_points.append(frame_list[-1] + 1)
    session_dfs = []
    for i in range(len(check_points) - 1):
        session_df = df[(df['index_col'] >= check_points[i]) & (df['index_col'] < check_points[i+1])]
        session_df.drop(columns='index_col', inplace=True)
        session_dfs.append(session_df)
    return session_dfs



In [5]:
# ----------------------------- 主程序 -----------------------------
print('开始处理...')

# 参数设置
ROI = '260427'
project = 'ITRT'
session_ids = [2]   # 需要分析的session编号
root_path = "F:/AAA-ZSY(5)"
data_path = os.path.join(root_path)
imgSave_path = os.path.join(root_path, 'Results')

mice_list = ['SYN18-ITRT', ]
mice_group = ['control'] * len(mice_list)

# 事件提取参数
gapMax = 10
intervalLength = 20
if_CombineAdjacentChunks = True
ifNorm = False
baseSecond = 2
beforeB = False
secF = 3          # 事件之前的时间 (秒)
secB = 8          # 事件之后的时间 (秒)
ifFilterOutCenter = False   # 改为False，因为现在分析所有神经元，无需过滤
ifFilterOutCorner = False
fps = 15
amgColor = '#3333F5'
amgFillColor = '#aaaafa'

# 导入自定义函数（请确保该文件存在于同一目录）
from Functions_ITRT_PCA_250811_checkframeinterval_beforeafter_eventonset import *



开始处理...


In [6]:


# ----------------------------- 第一步：生成带标签的session数据 -----------------------------
for i in range(len(mice_list)):
    mouseSummary = mice_list[i]
    mouseName = mouseSummary[-11:-7]
    mouseType = mice_group[i]
    print("Processing ", mouseName+'-'+mouseType)
    trace = os.path.join(root_path, mouseSummary, "TRACE_Con.csv")
    df_traces = separate_session(trace)
    for session_id in session_ids:
        event = os.path.join(root_path, mouseSummary, f"event{session_id}.xlsx")
        session = df_traces[session_id-1]
        session.index.name = "Frame"
        session.index = pd.to_numeric(session.index)
        session_frame_label = add_event_label(session, event)
        session_frame_label.to_csv(os.path.join(data_path, mouseSummary, f"session_{session_id}_trace.csv"))
        print(project, f"session_{session_id}_trace has shape:", session_frame_label.shape)
    print('Done')




# ----------------------------- 第二步：对每个session提取所有神经元的事件相关活性 -----------------------------
center_ceON_sessions = []   # free_allogroom
corner_ceON_sessions = []   # sniff
sniffg_ceON_sessions = []   # rear_selfgroom
cs_ceON_sessions = []       # all (全时段，通常无用)

for sessionID in session_ids:
    print(f'\nSession {sessionID}')
    center_ceON = pd.DataFrame()
    corner_ceON = pd.DataFrame()
    sniffg_ceON = pd.DataFrame()
    cs_ceON = pd.DataFrame()

    for i in range(len(mice_list)):
        mouseSummary = mice_list[i]
        mouseName = mouseSummary[-13:-7]
        mouseType = mice_group[i]

        # 读取已处理的session数据
        df = pd.read_csv(os.path.join(root_path, mouseSummary, f"session_{sessionID}_trace.csv"))
        df.columns = [c.strip() for c in df.columns]

        # 提取所有神经元列（排除Frame和Frame_Label）
        all_neurons = [col for col in df.columns if col not in ['Frame', 'Frame_Label']]
        print(f"{mouseSummary} session{sessionID} shape: {df.shape} | 神经元数量: {len(all_neurons)}")

        # 构建只包含神经元和标签的DataFrame
        new_df = df[['Frame'] + all_neurons + ['Frame_Label']]
        print(f"New session shape: {new_df.shape}")

        if len(all_neurons) > 0:
            try:
                # 处理 free_allogroom 事件
                Bcenter_ceON_list, centerA_ceON_list = get_BonsetA_debug(
                    new_df, 'free_allogroom', fps, secForward=secF, secBackward=secB,
                    beforeB=beforeB, baseSecond=baseSecond, intervalLength=intervalLength,
                    gapMax=gapMax, if_CombineAdjacentChunks=if_CombineAdjacentChunks, ifNorm=ifNorm)

                # 处理 sniff 事件
                Bcorner_ceON_list, cornerA_ceON_list = get_BonsetA_debug(
                    new_df, 'sniff', fps, secF, secB, beforeB,
                    baseSecond, intervalLength, if_CombineAdjacentChunks, gapMax, ifNorm=ifNorm)

                # 处理 rear_selfgroom 事件
                Bsniffg_ceON_list, sniffgA_ceON_list = get_BonsetA_debug(
                    new_df, 'rear_selfgroom', fps, secF, secB, beforeB,
                    baseSecond, intervalLength, if_CombineAdjacentChunks, gapMax, ifNorm=ifNorm)

                # 处理 all 事件（全时段，通常无有效区间）
                Bcs_ceON_list, csA_ceON_list = get_BonsetA_debug(
                    new_df, 'all', fps, secF, secB, beforeB,
                    baseSecond, intervalLength, if_CombineAdjacentChunks, gapMax, ifNorm=ifNorm)

                # 聚合每个事件类型的数据
                # 1. free_allogroom
                if len(Bcenter_ceON_list) > 0 and len(centerA_ceON_list) > 0:
                    Bcenter_ceON_Mean = average_nActEntries_debug(Bcenter_ceON_list, all_neurons, fps, secF)
                    centerA_ceON_Mean = average_nActEntries_debug(centerA_ceON_list, all_neurons, fps, secB)
                    BcenterA_ceON_Mean = pd.concat([Bcenter_ceON_Mean, centerA_ceON_Mean])
                    if ifFilterOutCenter:
                        BcenterA_ceON_Mean = filterOut_nonceONce(BcenterA_ceON_Mean, fps, secF)
                    if not BcenterA_ceON_Mean.empty:
                        center_ceON = pd.concat([center_ceON, BcenterA_ceON_Mean], axis=1)
                        print(f"  center shape: {center_ceON.shape}")

                # 2. sniff
                if len(Bcorner_ceON_list) > 0 and len(cornerA_ceON_list) > 0:
                    Bcorner_ceON_Mean = average_nActEntries_debug(Bcorner_ceON_list, all_neurons, fps, secF)
                    cornerA_ceON_Mean = average_nActEntries_debug(cornerA_ceON_list, all_neurons, fps, secB)
                    BcornerA_ceON_Mean = pd.concat([Bcorner_ceON_Mean, cornerA_ceON_Mean])
                    if ifFilterOutCorner:
                        BcornerA_ceON_Mean = filterOut_nonceONco(BcornerA_ceON_Mean, fps, secF)
                    if not BcornerA_ceON_Mean.empty:
                        corner_ceON = pd.concat([corner_ceON, BcornerA_ceON_Mean], axis=1)
                        print(f"  corner shape: {corner_ceON.shape}")

                # 3. rear_selfgroom
                if len(Bsniffg_ceON_list) > 0 and len(sniffgA_ceON_list) > 0:
                    Bsniffg_ceON_Mean = average_nActEntries_debug(Bsniffg_ceON_list, all_neurons, fps, secF)
                    sniffgA_ceON_Mean = average_nActEntries_debug(sniffgA_ceON_list, all_neurons, fps, secB)
                    BsniffgA_ceON_Mean = pd.concat([Bsniffg_ceON_Mean, sniffgA_ceON_Mean])
                    if not BsniffgA_ceON_Mean.empty:
                        sniffg_ceON = pd.concat([sniffg_ceON, BsniffgA_ceON_Mean], axis=1)
                        print(f"  freezing shape: {sniffg_ceON.shape}")

                # 4. all (通常为空，忽略)
                if len(Bcs_ceON_list) > 0 and len(csA_ceON_list) > 0:
                    Bcs_ceON_Mean = average_nActEntries_debug(Bcs_ceON_list, all_neurons, fps, secF)
                    csA_ceON_Mean = average_nActEntries_debug(csA_ceON_list, all_neurons, fps, secB)
                    BcsA_ceON_Mean = pd.concat([Bcs_ceON_Mean, csA_ceON_Mean])
                    if not BcsA_ceON_Mean.empty:
                        cs_ceON = pd.concat([cs_ceON, BcsA_ceON_Mean], axis=1)
                        print(f"  cs shape: {cs_ceON.shape}")

            except Exception as e:
                print(f"  处理过程中发生错误: {e}")
                import traceback
                traceback.print_exc()
                continue

    # 重置索引
    center_ceON = center_ceON.reset_index(drop=True)
    corner_ceON = corner_ceON.reset_index(drop=True)
    sniffg_ceON = sniffg_ceON.reset_index(drop=True)
    cs_ceON = cs_ceON.reset_index(drop=True)

    print(f'Session {sessionID} Done, with shapes:',
          center_ceON.shape, corner_ceON.shape, sniffg_ceON.shape, cs_ceON.shape)

    center_ceON_sessions.append(center_ceON)
    corner_ceON_sessions.append(corner_ceON)
    sniffg_ceON_sessions.append(sniffg_ceON)
    cs_ceON_sessions.append(cs_ceON)

print('所有session数据提取完成')

# ----------------------------- 第三步：绘制热图和平均轨迹 -----------------------------
cmap = 'coolwarm'
ifCenter = False

def sort_neurons_by_weighted_peak_time(df, window_size=5):
    """按加权峰值时间对神经元列进行排序，使热图更平滑"""
    if df.empty:
        return df
    peak_scores = {}
    for col in df.columns:
        activity = df[col].values
        weighted_scores = []
        for j in range(len(activity)):
            start = max(0, j - window_size)
            end = min(len(activity), j + window_size + 1)
            window_sum = np.sum(activity[start:end])
            weighted_scores.append(window_sum)
        peak_idx = np.argmax(weighted_scores)
        peak_scores[col] = peak_idx
    sorted_columns = sorted(peak_scores.items(), key=lambda x: x[1])
    sorted_columns = [col for col, _ in sorted_columns]
    return df[sorted_columns]

for i in range(len(session_ids)):
    sessionID = session_ids[i]
    print(f"\n处理Session {sessionID}")

    center_ceON = center_ceON_sessions[i]
    corner_ceON = corner_ceON_sessions[i]
    sniffg_ceON = sniffg_ceON_sessions[i]
    cs_ceON = cs_ceON_sessions[i]

    # 数据归一化
    normed_center = pd.DataFrame()
    normed_corner = pd.DataFrame()
    normed_sniffg = pd.DataFrame()
    normed_cs = pd.DataFrame()

    if not center_ceON.empty:
        print("处理 free_allogroom 数据...")
        normed_center = mixMaxScale_dff(center_ceON, (-1, 4))
    if not corner_ceON.empty:
        print("处理 sniff 数据...")
        normed_corner = mixMaxScale_dff(corner_ceON, (-1, 4))
    if not sniffg_ceON.empty:
        print("处理 rear_selfgroom 数据...")
        normed_sniffg = mixMaxScale_dff(sniffg_ceON, (-1, 4))
    if not cs_ceON.empty:
        print("处理 all 数据...")
        normed_cs = mixMaxScale_dff(cs_ceON, (-1, 4))

    # 排序神经元（按峰值时间）
    if not normed_center.empty:
        normed_center = sort_neurons_by_weighted_peak_time(normed_center, window_size=5)
    if not normed_corner.empty:
        normed_corner = sort_neurons_by_weighted_peak_time(normed_corner, window_size=5)
    if not normed_sniffg.empty:
        normed_sniffg = sort_neurons_by_weighted_peak_time(normed_sniffg, window_size=5)
    if not normed_cs.empty:
        normed_cs = sort_neurons_by_weighted_peak_time(normed_cs, window_size=5)

    # 绘制热图
    if not normed_center.empty:
        plot_colomap(normed_center, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path,
                     filename=f'heatmap_session{sessionID}_ALLNEURONS_free_allogroom_sorted')
    if not normed_corner.empty:
        plot_colomap(normed_corner, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path,
                     filename=f'heatmap_session{sessionID}_ALLNEURONS_sniff_sorted')
    if not normed_sniffg.empty:
        plot_colomap(normed_sniffg, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path,
                     filename=f'heatmap_session{sessionID}_ALLNEURONS_rear_selfgroom_sorted')
    if not normed_cs.empty:
        plot_colomap(normed_cs, ifCenter=ifCenter, cmap=cmap, vmin=-1, vmax=4,
                     ifSave=True, savePath=imgSave_path,
                     filename=f'heatmap_session{sessionID}_ALLNEURONS_all_sorted')

    # 绘制平均轨迹
    if not normed_center.empty:
        summary = getSummary(normed_center)
        x, y, y_sem_low, y_sem_high, vmin, vmax = get_minMaxForPlot_One(summary)
        plotAgeTrace_One(x, y, y_sem_low, y_sem_high, vmin, vmax,
                         xVline=secF*fps, cWT=amgColor, cFWT=amgFillColor,
                         ifSave=True, savePath=imgSave_path,
                         filename=f'avgTrace_session{sessionID}_ALLNEURONS_free_allogroom')

    if not normed_corner.empty:
        summary = getSummary(normed_corner)
        x, y, y_sem_low, y_sem_high, vmin, vmax = get_minMaxForPlot_One(summary)
        plotAgeTrace_One(x, y, y_sem_low, y_sem_high, vmin-0.3, vmax,
                         xVline=secF*fps, cWT=amgColor, cFWT=amgFillColor,
                         ifSave=True, savePath=imgSave_path,
                         filename=f'avgTrace_session{sessionID}_ALLNEURONS_sniff')

    if not normed_sniffg.empty:
        summary = getSummary(normed_sniffg)
        x, y, y_sem_low, y_sem_high, vmin, vmax = get_minMaxForPlot_One(summary)
        plotAgeTrace_One(x, y, y_sem_low, y_sem_high, vmin, vmax,
                         xVline=secF*fps, cWT=amgColor, cFWT=amgFillColor,
                         ifSave=True, savePath=imgSave_path,
                         filename=f'avgTrace_session{sessionID}_ALLNEURONS_rear_selfgroom')

    if not normed_cs.empty:
        summary = getSummary(normed_cs)
        x, y, y_sem_low, y_sem_high, vmin, vmax = get_minMaxForPlot_One(summary)
        plotAgeTrace_One(x, y, y_sem_low, y_sem_high, vmin, vmax,
                         xVline=secF*fps, cWT=amgColor, cFWT=amgFillColor,
                         ifSave=True, savePath=imgSave_path,
                         filename=f'avgTrace_session{sessionID}_ALLNEURONS_all')

    print(f"图像已保存至: {imgSave_path}")

print("全部处理完成！")

Processing  SYN-control
ITRT session_2_trace has shape: (3585, 77)
Done

Session 2
SYN18-ITRT session2 shape: (3585, 78) | 神经元数量: 76
New session shape: (3585, 78)
the shape of the check frame interval before and after onset is: (3585, 78)
the shape of the check frame interval before and after onset is: (3585, 78)
[[389, 414], [431, 454], [785, 811], [909, 932], [1447, 1467], [1545, 1572], [1590, 1613], [1937, 1972], [2145, 2174], [2671, 2692], [2834, 2871]]
  free_allogroom: found 11 intervals
  First interval: [389, 414], df length: 3585
  Valid intervals found: 11 before, 11 after
the shape of the check frame interval before and after onset is: (3585, 78)
the shape of the check frame interval before and after onset is: (3585, 78)
[[133, 192], [268, 342], [433, 477], [628, 732], [853, 882], [1183, 1302], [1919, 1993], [2024, 2068], [2294, 2338], [2849, 2908], [3134, 3178]]
  sniff: found 11 intervals
  First interval: [133, 192], df length: 3585
  Valid intervals found: 11 before, 11 